In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)

In [2]:
PROJECT_DIR = "/Users/hoyin/Desktop/DanaFarber/workspaces/CMI_Painter_Angiosarcoma_WES_analysis_mh_regional/scripts"
RAW_DATA_DIR = f"{PROJECT_DIR}/data/raw"
PROCESSED_DATA_DIR = f"{PROJECT_DIR}/data/processed"


## General formatting of other cancers

In [8]:
other_cancer_ehr_df = pd.read_excel(f"{RAW_DATA_DIR}/Batches_1-12_Dates_Notes_Masked (4).xlsx",sheet_name="Other Cancers")
other_cancer_ehr_df["STUDY ID"] = other_cancer_ehr_df["STUDY ID"].apply(lambda x: x.replace("ASCPROJECT","ASCProject"))
other_cancer_ehr_df.columns = [col.strip() for col in other_cancer_ehr_df.columns]
other_cancer_ehr_df


,STUDY ID,OTHER CANCERS,SITE OTHER CANCERS,DATE OTHER CANCERS (DAYS FROM DX)
0,ASCProject_0044,NaN,NaN,NaN
1,ASCProject_0066,LARGE B-CELL NON-HODGKINS LYMPHOMA,MEDIASTINAL LYMPH NODES,-1801
2,ASCProject_0067,NaN,NaN,NaN
3,ASCProject_0164,INVASIVE BREAST CANCER,LEFT BREAST,-2364
4,ASCProject_0179,RECTAL CARCINOID,RECTUM,1475
...,...,...,...,...
211,ASCProject_0417,NO,NaN,NaN
212,ASCProject_0427,HODGKIN'S DISEASE NODULAR SCLEROSING,THYMIC TUMOR AND RIGHT PLEURAL SATELLITE LESION,-6649
213,ASCProject_0434,PROSTATE CANCER,PROSTATE,-1861
214,ASCProject_0434,BASAL CELL CARCINOMA,"LEFT SHOULDER, BOTH LEGS, RIGHT UPPER CHEST",NOT FOUND IN RECORD


## Reclassify into broader subgroups

In [7]:
other_cancer_ehr_df["OTHER CANCERS"].value_counts()

INVASIVE BREAST CANCER                  35
BASAL CELL CARCINOMA                    19
NOT FOUND IN RECORD                     14
SQUAMOUS CELL CARCINOMA                 12
NO                                      10
IN SITU BREAST CANCER                    8
MELANOMA                                 3
ADENOCARCINOMA                           3
SQUAMOUS CELL CARCINOMA IN SITU          3
PAPILLARY THYROID CARCINOMA              3
PROSTATE CANCER                          3
ENDOMETRIAL CANCER                       3
TRANSITIONAL-CELL CARCINOMA              2
HODGKIN'S LYMPHOMA                       2
PHYLLODE TUMOR                           2
MELANOMA IN SITU                         2
LARGE B-CELL NON-HODGKINS LYMPHOMA       1
FOLLICULAR LYMPHOMA                      1
RECTAL CARCINOID                         1
PANCREATIC CANCER                        1
STROMAL TUMOR                            1
CANCER NOS                               1
CARCINOMA                                1
SQUAMOUS CE

In [38]:
subgroup_mapping = {
    "INVASIVE BREAST CANCER": "BREAST CANCER",
    "IN SITU BREAST CANCER": "BREAST CANCER",
    "INVASIVE DUCTAL CARCINOMA": "BREAST CANCER",
    "INVASIVE DUCTAL CARCINOMA W/ DCIS": "BREAST CANCER",
    "BREAST CANCER": "BREAST CANCER",
    "BREAST CANCER (NOS)": "BREAST CANCER",
    "BREAST CANCER NOS": "BREAST CANCER",
    
    "BASAL CELL CARCINOMA": "SKIN CANCER",
    "SQUAMOUS CELL CARCINOMA": "SKIN CANCER",
    "SQUAMOUS CELL CARCINOMA IN SITU": "SKIN CANCER",
    "MELANOMA": "SKIN CANCER",
    "MELANOMA IN SITU": "SKIN CANCER",
    
    "HODGKIN'S LYMPHOMA": "LYMPHOMA",
    "LARGE B-CELL NON-HODGKINS LYMPHOMA": "LYMPHOMA",
    "FOLLICULAR LYMPHOMA": "LYMPHOMA",
    "CHRONIC LYMPHOCYTIC LEUKEMIA": "LYMPHOMA",
    "HODGKIN'S DISEASE NODULAR SCLEROSING": "LYMPHOMA",
    
    "PHYLLODE TUMOR": "SARCOMA",
    "LEIOMYOSARCOMA": "SARCOMA",
    "STROMAL TUMOR": "SARCOMA",
    
        
    "COLON CANCER": "GASTROINTESTINAL CANCER",
    "PANCREATIC CANCER": "GASTROINTESTINAL CANCER",
    "RECTAL CARCINOID": "GASTROINTESTINAL CANCER",
    "CARCINOID TUMOR": "GASTROINTESTINAL CANCER",
    
    #"PROSTATE CANCER": "PROSTATE CANCER",
    #"ENDOMETRIAL CANCER": "ENDOMETRIAL CANCER",
    #"OVARIAN CARCINOMA": "OVARIAN CANCER",
    #"CANCER NOS": "CANCER (NOS)",
    #"CARCINOMA": "CANCER (NOS)",
    
    #"PROSTATE CANCER": "OTHER CANCERS",
    #"ENDOMETRIAL CANCER": "OTHER CANCERS",
    "OVARIAN CARCINOMA": "OTHER CANCERS",
    "ADENOCARCINOMA": "OTHER CANCERS",
    #"PAPILLARY THYROID CARCINOMA": "OTHER CANCERS",
    "TRANSITIONAL-CELL CARCINOMA": "OTHER CANCERS",
    "CANCER (NOS)": "OTHER CANCERS",
    "CANCER NOS": "OTHER CANCERS",
    "CARCINOMA": "OTHER CANCERS",
    "RENAL ONCOCYTOMA": 'OTHER CANCERS',
    "MYELODYPLASTIC SYNDROME": "OTHER CANCERS",
    
    "NOT FOUND IN RECORD": "NOT FOUND",
    "NO": "NOT FOUND"
}

other_cancer_ehr_df["OTHER CANCERS REMAPPED"] = other_cancer_ehr_df["OTHER CANCERS"].fillna("NOT FOUND")
other_cancer_ehr_df["OTHER CANCERS REMAPPED"] = other_cancer_ehr_df["OTHER CANCERS REMAPPED"].apply(lambda x: x.strip())
other_cancer_ehr_df["OTHER CANCERS REMAPPED"] = other_cancer_ehr_df["OTHER CANCERS REMAPPED"].replace(subgroup_mapping)
#other_cancer_ehr_df["OTHER CANCERS REMAPPED"] = other_cancer_ehr_df["OTHER CANCERS REMAPPED"].fillna("NOT FOUND")
other_cancer_ehr_df["OTHER CANCERS REMAPPED"].value_counts()



NOT FOUND                      96
BREAST CANCER                  47
SKIN CANCER                    40
OTHER CANCERS                  10
LYMPHOMA                        6
GASTROINTESTINAL CANCER         4
SARCOMA                         4
ENDOMETRIAL CANCER              3
PAPILLARY THYROID CARCINOMA     3
PROSTATE CANCER                 3
Name: OTHER CANCERS REMAPPED, dtype: int64

In [39]:
other_cancer_ehr_df["DATE OTHER CANCERS (DAYS FROM DX)"] = other_cancer_ehr_df["DATE OTHER CANCERS (DAYS FROM DX)"].replace({
    "NOT FOUND IN RECORD": np.nan
})

other_cancer_ehr_df

,STUDY ID,OTHER CANCERS,SITE OTHER CANCERS,DATE OTHER CANCERS (DAYS FROM DX),OTHER CANCERS REMAPPED
0,ASCProject_0044,NaN,NaN,NaN,NOT FOUND
1,ASCProject_0066,LARGE B-CELL NON-HODGKINS LYMPHOMA,MEDIASTINAL LYMPH NODES,-1801.0,LYMPHOMA
2,ASCProject_0067,NaN,NaN,NaN,NOT FOUND
3,ASCProject_0164,INVASIVE BREAST CANCER,LEFT BREAST,-2364.0,BREAST CANCER
4,ASCProject_0179,RECTAL CARCINOID,RECTUM,1475.0,GASTROINTESTINAL CANCER
...,...,...,...,...,...
211,ASCProject_0417,NO,NaN,NaN,NOT FOUND
212,ASCProject_0427,HODGKIN'S DISEASE NODULAR SCLEROSING,THYMIC TUMOR AND RIGHT PLEURAL SATELLITE LESION,-6649.0,LYMPHOMA
213,ASCProject_0434,PROSTATE CANCER,PROSTATE,-1861.0,PROSTATE CANCER
214,ASCProject_0434,BASAL CELL CARCINOMA,"LEFT SHOULDER, BOTH LEGS, RIGHT UPPER CHEST",NaN,SKIN CANCER


In [42]:
prior_cancer_only = other_cancer_ehr_df[
    (~(other_cancer_ehr_df["DATE OTHER CANCERS (DAYS FROM DX)"] > 0)) |
    (other_cancer_ehr_df["OTHER CANCERS"] == "NOT FOUND")
]
prior_cancer_only["OTHER CANCERS REMAPPED"].value_counts()

NOT FOUND                      96
BREAST CANCER                  44
SKIN CANCER                    31
OTHER CANCERS                   6
LYMPHOMA                        4
ENDOMETRIAL CANCER              3
PAPILLARY THYROID CARCINOMA     3
PROSTATE CANCER                 3
GASTROINTESTINAL CANCER         2
SARCOMA                         2
Name: OTHER CANCERS REMAPPED, dtype: int64

In [41]:
other_cancer_ehr_df[
    other_cancer_ehr_df["DATE OTHER CANCERS (DAYS FROM DX)"] > 0
]

,STUDY ID,OTHER CANCERS,SITE OTHER CANCERS,DATE OTHER CANCERS (DAYS FROM DX),OTHER CANCERS REMAPPED
4,ASCProject_0179,RECTAL CARCINOID,RECTUM,1475.0,GASTROINTESTINAL CANCER
5,ASCProject_0015,FOLLICULAR LYMPHOMA,LEFT SUMANDIBULAR NODE,1448.0,LYMPHOMA
16,ASCProject_0277,SQUAMOUS CELL CARCINOMA IN SITU,RIGHT TEMPLE SKIN,2057.0,SKIN CANCER
30,ASCProject_0340,PANCREATIC CANCER,PANCREAS,1066.0,GASTROINTESTINAL CANCER
46,ASCProject_0035,RENAL ONCOCYTOMA,KIDNEY,1773.0,OTHER CANCERS
48,ASCProject_0121,STROMAL TUMOR,GASTROINTESTINAL TRACT,2268.0,SARCOMA
63,ASCProject_0003,LEIOMYOSARCOMA,PERITONEUM,256.0,SARCOMA
64,ASCProject_0003,ADENOCARCINOMA,"RECTUM, LUNG, LIVER, BONE",953.0,OTHER CANCERS
67,ASCProject_0007,CARCINOMA,THYROID,58.0,OTHER CANCERS
87,ASCProject_0097,SQUAMOUS CELL CARCINOMA,LEFT EAR,1035.0,SKIN CANCER


In [44]:
other_cancer_ehr_df.to_csv(f"{PROCESSED_DATA_DIR}/other_cancer_data.tsv",sep="\t",index=False)